# Part D: one controlled loss ablation

**Part C result:** no ensemble passed the paired improvement gate. The reference remains
`ld_large:baseline` (58.93 LD span F1). The best own ensemble reached 57.34,
difference -1.59 [95% CI -4.10, +1.20]. Tuned soft targets were slightly below hard
targets in all three matched seeds. These are the reported results, not new measurements.

This notebook tests **one change**: answer-token cross-entropy **plus 0.2 times a
character-weighted soft-F1 loss**. Keep ModernBERT-base, hard targets, seed 42,
3 epochs, learning rate 2e-5, 4,096 context and an 8,192-token batch budget.
This is a single prespecified candidate; do not sweep the loss weight on test results.

Push the new local commits first. Select a **T4 GPU**, then run cells in order.
The training-and-evaluation cell has a **110-minute total time budget**. On timeout or
disconnect, rerun it to resume from Drive. Completed steps are verified and skipped.
Runtime on T4 is not yet measured; the timer bounds each attempt, not the work needed.
Do not run two experiments against the same checkpoint at once.


## 1. Connect Drive and get the code

Let any earlier notebook finish before updating this checkout. Saved predictions,
models and result files remain in `MyDrive/hallu_poc/work`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, time
from pathlib import Path
from IPython.display import Markdown, display

REPO = Path('/content/tse')
BRANCH = 'fix/runnable-pipeline'
def setup_command(command):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    result.check_returncode()

if not REPO.exists():
    setup_command(['git', 'clone', '--single-branch', '--branch', BRANCH,
                   'https://github.com/gbirhanu/tse.git', str(REPO)])
else:
    actual = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
    assert actual == BRANCH, f'Expected {BRANCH}, found {actual}'
    setup_command(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH])
PROJECT = REPO / 'hallu_poc'
WORK = Path('/content/drive/MyDrive/hallu_poc/work')
DATA = Path('/content/drive/MyDrive/hallu_poc/RAGTruth')
if not (DATA / 'response.jsonl').exists() and not (DATA / 'dataset/response.jsonl').exists():
    DATA = Path('/content/RAGTruth')
    if not DATA.exists():
        setup_command(['git', 'clone', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth.git', str(DATA)])
setup_command([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'tqdm>=4.27,<5'])
REVISION = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Code revision:', REVISION)
print('Saved outputs:', WORK)


## 2. Show existing results and check the comparison files

Result tables are rendered explicitly in this notebook. Progress updates show
elapsed time and ETA for the displayed phase. This cell does not train a model.


In [ ]:
sys.path.insert(0, str(PROJECT))
import importlib
import hallu.notebook
importlib.reload(hallu.notebook)
from hallu.notebook import run_command

def run(*arguments, timeout=110 * 60):
    command = [sys.executable, '-u', str(PROJECT / 'poc.py'), *map(str, arguments),
               '--data-dir', str(DATA), '--work-dir', str(WORK)]
    environment = {**os.environ, 'TOKENIZERS_PARALLELISM': 'false'}
    try:
        if arguments[0] == 'report':
            result = subprocess.run(command, cwd=PROJECT, timeout=timeout, env=environment,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            if result.returncode:
                print(result.stdout, flush=True)
                result.check_returncode()
            display(Markdown(result.stdout))
        else:
            result = run_command(command, cwd=PROJECT, env=environment, timeout=timeout)
            table = '\n'.join(line for line in result.stdout.splitlines() if line.startswith('|'))
            if table:
                display(Markdown(table))
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError('Time budget reached. Checkpoints are on Drive; rerun this cell to resume.') from exc

help_text = subprocess.check_output([sys.executable, str(PROJECT / 'poc.py'), '--help'], text=True)
assert '--span-f1-weight' in help_text, 'Push the Part D commits, then rerun setup.'
required = ['preds/ld_large_test.pkl', 'preds/ld_large_spans.pkl', 'results/ld_large_levers.json',
            'preds/mb_hard_test.pkl', 'preds/mb_hard_spans.pkl', 'results/mb_hard_levers.json',
            'results/mb_hard_training.json']
missing = [name for name in required if not (WORK / name).exists()]
assert not missing, f'Missing saved comparison files: {missing}. Use the original Part A/B work folder.'
RUN_NAME = 'mb_hard_f1w020_s42'
print('Candidate:', RUN_NAME, '| Fixed loss weight: 0.2 | Reference: ld_large:baseline')
run('report')


## 3. Train, tune on dev, and evaluate the one candidate

This cell resumes training/prediction if interrupted and skips completed work.
It compares the candidate with **LettuceDetect**, the retained reference.
Decoder thresholds, token-gap merging and word snapping are fitted only on dev.
A positive gain with a strictly positive lower paired 95% bound is required to keep it.

The new term is a **surrogate**: soft character overlap over covered answer characters.
Character contributions are shared across overlapping token offsets; gold overlaps count
once. All-negative batches use mean predicted hallucination probability. It does not
exactly reproduce the LD scorer's overlapping spans or the decoder's merging/snapping.
CE is unchanged; reported training loss is the combined objective. Checkpoint selection
remains dev LD span F1 at the original 0.5 threshold, exactly as in the control.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before training.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
deadline = time.monotonic() + 110 * 60
def remaining():
    seconds = deadline - time.monotonic()
    if seconds <= 0:
        raise RuntimeError('110-minute budget reached. Rerun this cell to resume.')
    return seconds

run('train', '--name', RUN_NAME, '--base', 'answerdotai/ModernBERT-base',
    '--targets', 'hard', '--seed', '42', '--epochs', '3', '--lr', '2e-5',
    '--max-length', '4096', '--batch-tokens', '8192', '--checkpoint-every', '200',
    '--span-f1-weight', '0.2', '--device', 'cuda', timeout=remaining())
run('levers', '--name', RUN_NAME, '--fit', 'dev',
    '--against', 'ld_large:baseline', timeout=remaining())


## 4. Compare with the matched hard-target control

This is a different question: did the new loss improve on the same seed-42 model?
The tuned control is 55.48. The output says **candidate minus control**, with a paired
interval from saved per-answer predictions. A control gain alone does not replace
LettuceDetect; use the preceding cell's reference comparison for that decision.
A single seed is a screening experiment, not a robustness claim.
The legacy seed-42 control has no recorded dependency/model revision: settings
match its documented run, but identical software cannot be verified. Any promising
result needs replication with a freshly trained control before a paper-level claim.


In [ ]:
run('report', '--compare', 'mb_hard', RUN_NAME, timeout=10 * 60)


## 5. Display and save the final table again

Run this cell whenever you need the results. It does not train anything.
Paste the table and the candidate-minus-control differences from cell 4 back into the chat.
If this runtime was reset, first rerun setup and the helper cell; no GPU is needed for this report.


In [ ]:
command = [sys.executable, '-u', str(PROJECT / 'poc.py'), 'report', '--work-dir', str(WORK)]
report = subprocess.run(command, cwd=PROJECT, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, timeout=120)
if report.returncode:
    print(report.stdout)
    report.check_returncode()
display(Markdown(report.stdout))
report_path = WORK / 'results' / 'part_d_report.txt'
report_path.write_text(report.stdout, encoding='utf-8')
print('Table saved to:', report_path)
